In [1]:
import geopandas as gpd
import pandas as pd
import os

# --- 1. Path ไฟล์ POI ---
poi_paths = [
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\clinic\clinic.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\convenience\convenience_store.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\education\school.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\education\university.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\gov\gov_service.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\hospital\Hospital.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\mall\mall.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\night_clue\night_club.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\public_park\public_park.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\restaurant\restaurant.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\supermarket\supermarket.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\transpot\airport\Airport.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\transpot\e_railway\E_railway.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\poi_data\transpot\railway\railway.shp"
]

# --- 2. Path ไฟล์ Buffer ---
buffer_paths = [
    r"C:\Users\Asus\Desktop\ingest_data\gis_data\5marge_range\merge_200m\merge_200m.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\gis_data\5marge_range\merge_500m\merge_500m.shp", 
    r"C:\Users\Asus\Desktop\ingest_data\gis_data\5marge_range\merge_1000m\merge_1000m.shp"
]

# List สำหรับเก็บ DataFrame ของแต่ละระยะ
dfs_by_range = []

print("กำลังเริ่มประมวลผล (รวมคอลัมน์ในบรรทัดเดียว)...")

for i, buf_path in enumerate(buffer_paths):
    if not os.path.exists(buf_path):
        print(f"Error: หาไฟล์ไม่เจอ -> {buf_path}")
        continue

    # อ่านไฟล์ Buffer
    gdf_buffer = gpd.read_file(buf_path)
    
    # ดึงชื่อระยะทางจากชื่อไฟล์ (เช่น merge_200m -> 200m)
    # หรือใช้วิธี split string เอา
    filename = os.path.basename(buf_path)
    if "200m" in filename: range_suffix = "_200m"
    elif "500m" in filename: range_suffix = "_500m"
    elif "1000m" in filename: range_suffix = "_1000m"
    else: range_suffix = f"_file_{i+1}" # กรณีชื่อไม่ตรง pattern

    print(f"กำลังทำระยะ: {range_suffix.replace('_','')} (จำนวน {len(gdf_buffer)} แถว)...")

    # สร้าง DataFrame ชั่วคราวสำหรับระยะนี้
    # ถ้าเป็นไฟล์แรก (200m) เราจะเก็บ attribute เดิมไว้ด้วย (เผื่อมีชื่อคอนโด)
    # ถ้าเป็นไฟล์หลังๆ เราจะเก็บแค่ index เพื่อเอาไว้ Join
    if i == 0:
        # เก็บทุกคอลัมน์ที่ไม่ใช่ geometry จากไฟล์แรก
        df_current = pd.DataFrame(gdf_buffer.drop(columns='geometry'))
    else:
        # ไฟล์ถัดๆ ไป สร้างตารางเปล่าที่มีแค่ index
        df_current = pd.DataFrame(index=gdf_buffer.index)

    # วนลูป POI
    for poi_path in poi_paths:
        poi_base_name = os.path.basename(poi_path).replace('.shp', '')
        
        # ตั้งชื่อคอลัมน์ใหม่ เช่น clinic_200m, clinic_500m
        new_col_name = f"{poi_base_name}{range_suffix}"
        
        if not os.path.exists(poi_path):
            df_current[new_col_name] = 0
            continue
            
        try:
            gdf_poi = gpd.read_file(poi_path)
            if gdf_buffer.crs != gdf_poi.crs:
                gdf_poi = gdf_poi.to_crs(gdf_buffer.crs)

            # Spatial Join
            joined = gpd.sjoin(gdf_poi, gdf_buffer, predicate='within')
            
            # นับจำนวน group by index ของ buffer
            counts = joined.groupby('index_right').size()
            
            # Map ค่าลงไปใน DataFrame
            df_current[new_col_name] = df_current.index.map(counts).fillna(0).astype(int)
            
        except Exception as e:
            print(f"  - Error ({poi_base_name}): {e}")
            df_current[new_col_name] = 0
    
    # เก็บผลลัพธ์ของระยะนี้เข้า List
    dfs_by_range.append(df_current)

# --- 3. รวมร่าง (Merge) ---
if dfs_by_range:
    print("\nกำลังรวมตาราง...")
    
    # รวมตารางแบบ "ต่อด้านข้าง" (axis=1)
    # สมมติฐาน: ไฟล์ทั้ง 3 เรียงลำดับ row มาเหมือนกันเป๊ะ
    final_df = pd.concat(dfs_by_range, axis=1)
    
    # ลบคอลัมน์ที่ชื่อซ้ำกัน (ถ้ามี)
    final_df = final_df.loc[:,~final_df.columns.duplicated()]

    print("\n" + "="*50)
    print("ตัวอย่างผลลัพธ์ (5 แถวแรก):")
    print("="*50)
    print(final_df.head())

    # บันทึกไฟล์
    output_path = r"C:\Users\Asus\Desktop\ingest_data\tranfrom_data\poi_summary_condo_by_id.csv"
    final_df.to_csv(output_path, index=True) # เก็บ index ไว้เผื่อเช็ค row id
    print(f"\n[Success] บันทึกไฟล์เสร็จสิ้นที่:\n{output_path}")
else:
    print("เกิดข้อผิดพลาด ไม่สามารถประมวลผลได้")

กำลังเริ่มประมวลผล (รวมคอลัมน์ในบรรทัดเดียว)...
กำลังทำระยะ: 200m (จำนวน 1516 แถว)...
  - Error (Airport): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data
  - Error (E_railway): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data
  - Error (railway): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data


c:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from TIS-620 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(
c:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from TIS-620 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


กำลังทำระยะ: 500m (จำนวน 1516 แถว)...
  - Error (Airport): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data
  - Error (E_railway): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data
  - Error (railway): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data


c:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from TIS-620 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(
c:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from TIS-620 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


กำลังทำระยะ: 1000m (จำนวน 1516 แถว)...
  - Error (Airport): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data
  - Error (E_railway): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data
  - Error (railway): 'utf-8' codec can't decode byte 0xe0 in position 9: unexpected end of data

กำลังรวมตาราง...

ตัวอย่างผลลัพธ์ (5 แถวแรก):
     id                    Name        lat        long   source_fil  \
0     1  @ City Sukhumvit 101/1  13.685126  100.614427     id_1.shp   
1    10           39 by Sansiri  13.731767  100.570422    id_10.shp   
2   100              Baan Prida  13.738489  100.556278   id_100.shp   
3  1000    Siamese Sukhumvit 87  13.700531  100.603970  id_1000.shp   
4  1001        Siamese Surawong  13.730033  100.527984  id_1001.shp   

   clinic_200m  convenience_store_200m  school_200m  university_200m  \
0            1                       1            1                0   
1            1                       4            0  

c:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from TIS-620 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(
c:\Users\Asus\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: One or several characters couldn't be converted correctly from TIS-620 to UTF-8.  This warning will not be emitted anymore
  return ogr_read(
